# Synchronous FedAvg (notebook 01)

The classic baseline. Every round, **all** clients train on the same global model, and the server **waits for every one** before averaging.

**How it works**

1. All clients download the global model and train.
2. The server waits until **all** updates have arrived.
3. New global model = average of the client models, weighted by sample count.
4. After `round_gap` seconds, the next round starts.

> **Note:** there is no staleness, but the slowest client sets the pace of every round. That's the **straggler problem** async methods try to fix.

> **Important:** `copy_model_at = "dispatch"` doesn't change the model (it is frozen during a round anyway). It only keeps the exact random order of the original notebook.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class SyncFedAvg(Approach):
    name = "sync_fedavg"
    label = "Synchronous wait-for-all"          # shown in plot legends
    round_gap = 1.0               # idle virtual seconds between rounds
    copy_model_at = "dispatch"

    def start(self, server):
        self.round = []           # updates received in the current round
        for c in server.clients:
            server.train(c.id)

    def client_free(self, server, client):
        pass                      # don't restart: wait for the whole round

    def on_update(self, server, update):
        self.round.append(update)
        if len(self.round) == len(server.clients):             # everyone is back
            updates = sorted(self.round, key=lambda u: u.client_id)
            server.set_model(fedavg(updates), updates)
            self.round = []
            for c in server.clients:                           # next round
                server.train(c.id, at=server.time + self.round_gap)

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(SyncFedAvg(round_gap=0))`. It is saved as a separate run.

In [3]:
results = run(SyncFedAvg())

sync_fedavg                  seed=42   val= 94.52%  acc= 95.20%  time=  1394.5s  updates=  100  worst10%= 94.15%  wall=5.8s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
10,42,1394.459,100,0.952,0.153,0.9415,0.0
